# M64 A2 baseline and teacher qualification

Revision: **M64-TEACHER-QUALIFICATION-2026-10-04-r1**.

Keep A2 (MobileNetV4 Conv Medium + full MonoDETR) unchanged. Evaluate one candidate teacher, [MonoPRIO](https://github.com/Leon-Davies/MonoPRIO), before deciding whether it can teach our student. This notebook performs **zero optimizer updates**, not distillation or teacher training.

Use a **GPU runtime with CUDA 13.0**; an L4 or A100 is suitable for this batch-1 workflow. Run sections **1–6 in order**. No iPhone connection is needed. A2 epoch130, the original KITTI data, and Chen split files must remain on Drive. Official teacher validation weights/prior/log are downloaded automatically.

After a disconnect, run 1–6 again with the same RUN_ID on the same recorded environment. Verified prediction caches are reused. If the GPU or runtime identity changes, retain the old folder and select a new RUN_ID; do not edit old manifests. Section 6 can bundle failure logs even when an earlier section fails.


## 1 Settings and durable logging

Only change the paths below if your Drive layout differs. The notebook prints its revision so stale copies are easy to identify. It uses the project's main branch and fresh prospective model checkouts, never the frozen M62/M63 directories.


In [ ]:
# M64-TEACHER-QUALIFICATION-2026-10-04-r1 — self-contained, no historical runtime rollback
from pathlib import Path
from collections import deque
import json, os, shlex, subprocess, sys, runpy
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M64-TEACHER-QUALIFICATION-2026-10-04-r1'
RUN_ID = 'm64_teacher_qualification_r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_REPO = Path('/content/MonoDETR_M64_A2')
TEACHER_REPO = Path('/content/MonoPRIO_M64')
DATASET = Path('/content/kitti_m64')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
A2_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m64_teacher_qualification') / RUN_ID
PYTHON = Path('/content/m64_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
MANIFEST = OUTPUT/'m64_manifest.json'
A2_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
TEACHER_COMMIT = '884b7d8562e528031616c4773170bfc4fe211bf0'
SETUP_READY = MANIFEST_READY = SMOKE_READY = EVALUATION_READY = False
ENV = os.environ.copy()
OUTPUT.mkdir(parents=True, exist_ok=True)
print(REVISION, '\nOutput:', OUTPUT, flush=True)

def run_logged(command, name, cwd=None, allow_failure=False):
    command = [str(x) for x in command]
    log_path = OUTPUT/'colab_logs'/f'{name}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=60)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND: ' + shlex.join(command) + '\n')
        with subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=ENV,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush(); tail.append(line.rstrip())
            rc = process.wait()
        log.write(f'\nEXIT CODE: {rc}\n')
    if rc and not allow_failure:
        raise RuntimeError(f'Exit {rc}; full log={log_path}\n' + '\n'.join(tail))
    return rc

def checkout(path, url, commit):
    if '[' in url or ']' in url or not url.startswith('https://github.com/'):
        raise ValueError('Expected a plain public GitHub URL, not Markdown')
    if not path.exists():
        run_logged(['git', 'clone', '--no-checkout', url, path], f'clone_{path.name}')
        run_logged(['git', 'checkout', '--detach', commit], f'pin_{path.name}', path)
    if not (path/'.git').is_dir():
        raise RuntimeError(f'{path} is not a checkout; preserve it and choose a fresh model path')
    head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip()
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=path,text=True).strip()
    if head != commit or origin.rstrip('/') != url:
        raise RuntimeError(f'Unexpected model checkout: {path}, {head}, {origin}; do not reset it')

def qualify(stage, role=None, extra=(), allow_failure=False):
    command = [PYTHON, '-X', 'faulthandler', '-u',
               MOBILE_REPO/'scripts/m64_teacher_qualification.py', stage]
    command += ['--output', OUTPUT] if stage in ('assets','prepare','bundle') else ['--manifest',MANIFEST]
    if role: command += ['--role',role]
    command += list(extra)
    return run_logged(command, f'm64_{stage}_{role or "common"}', MOBILE_REPO, allow_failure)

if not MOBILE_REPO.exists():
    run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO], 'clone_mobile')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branch work')
run_logged(['git','pull','--ff-only'], 'update_mobile', MOBILE_REPO)
workflow = MOBILE_REPO/'scripts/m64_teacher_qualification.py'
if not workflow.is_file(): raise FileNotFoundError('M64 scripts missing. Use the published main-branch notebook.')
namespace = runpy.run_path(str(workflow), run_name='m64_revision_check')
if namespace['REVISION'] != REVISION:
    raise RuntimeError('Notebook and scripts have different revisions; reload the latest notebook')


## 2 Isolated runtime and pinned model sources

This creates a prospective private Python environment using virtualenv, not ensurepip. It does not downgrade the Colab kernel or reconstruct M62. Both attention extensions are built locally in separate checkouts. Model imports are patched so the KITTI AP evaluator is not initialized by training or inference.


In [ ]:
if not workflow.is_file(): raise RuntimeError('Run section 1 first')
helpers = runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'), run_name='m64_runtime_helpers')
ENV = helpers['runtime_env'](CUDA)
run_logged([sys.executable, '-u', MOBILE_REPO/'scripts/setup_m64_runtime.py',
            '--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',OUTPUT/'m64_runtime_receipt.json'],
           'm64_runtime', MOBILE_REPO)
checkout(A2_REPO, 'https://github.com/ZrrSkywalker/MonoDETR.git', A2_COMMIT)
checkout(TEACHER_REPO, 'https://github.com/Leon-Davies/MonoPRIO.git', TEACHER_COMMIT)
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON, MOBILE_REPO/'scripts'/patch, '--monodetr-repo', A2_REPO], patch.removesuffix('.py'), MOBILE_REPO)
for role, repo in (('a2',A2_REPO),('teacher',TEACHER_REPO)):
    run_logged([PYTHON, MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',repo,'--role',role],
               f'm64_patch_{role}', MOBILE_REPO)
    run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py',
                '--repo',repo,'--role',role], f'm64_build_{role}', MOBILE_REPO)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_m64_teacher_qualification.py','-v'],
           'm64_regression_tests', MOBILE_REPO)
SETUP_READY = True
print('Section 2 passed: isolated runtime and separate source-bound extension builds.')


## 3 Restore data and freeze this experiment

A2 is the original epoch130 checkpoint, verified by its saved hash. Teacher assets come from the pinned official validation links, not trainval/test weights. Downloads must match the release bytes inspected locally on 2026-10-04; no publisher-signed checksum is claimed. Normalized split IDs must match the exact Chen train/val order.


In [ ]:
if not SETUP_READY: raise RuntimeError('Run sections 1 and 2 first')
if not A2_SELECTION.is_file(): raise FileNotFoundError(A2_SELECTION)
if not SPLITS.is_dir(): raise FileNotFoundError(SPLITS)
candidates = ['/content/kitti_m62','/content/kitti_m61','/content/kitti',
              '/content/monodetr_kitti','/content/monodetr_kitti_a2',
              '/content/drive/MyDrive/datasets/kitti']
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in candidates: command += ['--candidate',candidate]
run_logged(command,'m64_restore_data',MOBILE_REPO)
qualify('assets')
qualify('prepare',extra=['--a2-repo',A2_REPO,'--teacher-repo',TEACHER_REPO,
                         '--dataset-root',DATASET,'--a2-selection',A2_SELECTION])
m = json.loads(MANIFEST.read_text())
assert m['optimizer_steps'] == 0 and m['kd_authorized'] is False
MANIFEST_READY = True
print('Section 3 passed. Fresh prospective identity frozen; historical manifests untouched.')


## 4 Model smoke checks with zero optimizer updates

A2: restricted checkpoint loading, finite inference, one real GT-loss backward, unchanged parameters and frozen BatchNorm buffers. Teacher: restricted checkpoint loading, finite inference, exact agreement between the downloaded prior bank and checkpoint buffers. The prior-bank training IDs are not independently embedded/verified; matching buffers alone does not prove absence of validation leakage.


In [ ]:
if not MANIFEST_READY: raise RuntimeError('Run sections 1–3 first')
SMOKE_READY = False
for role in ('a2','teacher'):
    qualify('smoke',role)
    report = json.loads((OUTPUT/f'm64_{role}_smoke.json').read_text())
    assert report['complete'] and report['finite_outputs'] and report['optimizer_steps'] == 0
    if role == 'a2':
        assert report['finite_nonzero_gradients'] and report['parameters_unchanged']
        assert report['running_buffers_unchanged']
    else:
        assert report['prior_buffers_match_checkpoint']
SMOKE_READY = True
print('Section 4 passed. No optimizer was created. No checkpoint or teacher was promoted.')


## 5 Complete validation and separate metric processes

Both models evaluate all 3,769 validation images. Progress prints every 100 images; each atomic per-image cache records the exact transformed input and can be verified/reused after interruption. Product Vehicle/Pedestrian metrics and native Car/Pedestrian metrics remain separate. Nearby recall here is 2D-IoU-matched diagnostic recall, not a 3D correctness rate.

The teacher's published threshold/protocol is also evaluated in its own process. If that native evaluator fails, its exit code/log remain in the bundle and the teacher is **not** marked reproduced. The completed prediction caches and independent CPU metrics are preserved.


In [ ]:
if not SMOKE_READY: raise RuntimeError('Run sections 1–4 first; do not bypass a failed smoke')
EVALUATION_READY = False
for role in ('a2','teacher'):
    qualify('infer',role)
    qualify('metrics',role)
native_rc = qualify('native',allow_failure=True)
native_status = dict(revision=REVISION,manifest_sha256=m['manifest_sha256'],
                     exit_code=native_rc,successful_exit=native_rc == 0,
                     log=str(OUTPUT/'colab_logs/m64_native_common.log'),
                     kd_authorized=False)
(OUTPUT/'m64_native_process.json').write_text(json.dumps(native_status,indent=2)+'\n')
if native_rc:
    print('Published native evaluation failed. Preserve the logs; no teacher selection or KD is authorized.')
EVALUATION_READY = True


## 6 Review bundle and stop

Return **m64_results.zip**. It contains identities, smoke reports, complete-set AP tables, nearby/geometry diagnostics, downloaded official log and durable execution logs; model weights and thousands of prediction files remain on Drive. A2 reproduction checks allow 0.15 AP points and 0.01 nearby recall numerical/protocol drift; these are not new product targets. Published native teacher reproduction allows 0.5 AP points per difficulty, fixed before execution.

We review baseline stability, actual per-class strengths, native/product taxonomy coverage and prior provenance before freezing any training pilot. This notebook cannot authorize distillation, teacher training, phone deployment, or a parameter sweep. If an earlier section failed, run this section to return a partial diagnostic bundle rather than repeatedly changing your environment.


In [ ]:
if EVALUATION_READY:
    qualify('review')
else:
    print('Bundling partial diagnostics only. Qualification is not complete.')
qualify('bundle')
print('Return:', OUTPUT/'m64_results.zip')
print('STOP FOR REVIEW. Next candidate step is A2-preservation control, then one matched feature-KD pilot.')
